# Fine-Grained 3-Way Parameter Grid Search

Systematic sweep over the three key model parameters:
- **σ₀** (encoding noise): noise applied once at memory insertion
- **σ** (diffusive noise): per-step noise during Langevin dynamics
- **η** (drift step size): magnitude of prior-driven drift per trial

For each (σ₀, σ, η) triple, compute d' at each ISI via Monte Carlo.
Visualize results as heatmaps for compact, interpretable summaries.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from src.model.analytic_gmm_2d import AnalyticGMM2D, make_default_gmm
from src.model.score_adapter_2d import ScoreAdapter2D
from utls.sandbox_2d_data import make_2d_grid_stimuli, compute_geometry_descriptors
from utls.runners_2d import run_model_core_2d
from utls.toy_experiments import make_high_diversity_sequences
from utls.roc_utils import roc_from_arrays, auroc_to_dprime
from utls.analysis_helpers import bootstrap_dprime_ci

SAVE_DIR = '../reports/figures/2d_grid_search'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Ready.')

## Setup: GMM prior, stimuli, sequences

In [ ]:
gmm = make_default_gmm()
X0, name_to_idx, stimulus_pool = make_2d_grid_stimuli()

# Unit-normalized score adapter
adapter = ScoreAdapter2D(gmm, normalize=True)

# Experiment sequences
ISI_VALUES = (0, 2, 16)
N_SEQUENCES = 10
SEQ_LENGTH = 81
MIN_PAIRS_PER_ISI = 5
N_MC = 10
SEED = 42
METRIC = 'cosine'

experiment_list, isi_keys = make_high_diversity_sequences(
    stimulus_pool=stimulus_pool,
    isi_values=list(ISI_VALUES),
    n_sequences=N_SEQUENCES,
    length=SEQ_LENGTH,
    min_pairs_per_isi=MIN_PAIRS_PER_ISI,
    seed=SEED,
)

print(f'{len(experiment_list)} sequences, length {len(experiment_list[0])}')
print(f'ISI values: {ISI_VALUES}')

## Define grid + MC sweep function

In [ ]:
# --- Fine-grained parameter grids ---
sigma0_grid = np.array([0.0, 0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0])
sigma_grid  = np.array([0.0, 0.025, 0.05, 0.1, 0.15, 0.2, 0.3])
eta_grid    = np.array([0.0, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2])

print(f'Grid size: {len(sigma0_grid)} x {len(sigma_grid)} x {len(eta_grid)} = '
      f'{len(sigma0_grid) * len(sigma_grid) * len(eta_grid)} configurations')
print(f'With {N_MC} MC reps each and {len(ISI_VALUES)} ISIs')


def run_mc_dprime(sigma0, sigma, eta, isi_values=ISI_VALUES, n_mc=N_MC, seed=SEED):
    """Run MC sweep and return d' per ISI as a dict."""
    runner_isi_values = [isi + 1 for isi in isi_values]
    score_type = 'distance'

    all_isi_hits = defaultdict(list)
    all_fas = []

    for rep in range(n_mc):
        run = run_model_core_2d(
            sigma0=sigma0, sigma=sigma,
            X0=X0, name_to_idx=name_to_idx,
            experiment_list=experiment_list,
            score_model=adapter,
            drift_step_size=eta,
            metric=METRIC,
            seed=seed * 10_000 + rep,
        )
        for risi in runner_isi_values:
            all_isi_hits[risi].extend(run['isi_hit_dists'].get(risi, []))
        all_fas.extend(run['fas'])

    all_fas = np.array(all_fas, dtype=float)
    result = {}

    for exp_isi, risi in zip(isi_values, runner_isi_values):
        hits_raw = all_isi_hits.get(risi, [])
        if len(hits_raw) < 3:
            result[exp_isi] = np.nan
            continue
        hits = np.array([s for s, t in hits_raw], dtype=float)
        roc = roc_from_arrays(hits, all_fas, score_type=score_type)
        if roc is None:
            result[exp_isi] = np.nan
        else:
            _, _, auc_val = roc
            result[exp_isi] = auroc_to_dprime(auc_val)

    return result

## Run the full grid search

In [ ]:
# Storage: results[isi][(i_s0, i_sig, i_eta)] = d'
results = {isi: np.full((len(sigma0_grid), len(sigma_grid), len(eta_grid)), np.nan)
           for isi in ISI_VALUES}

total = len(sigma0_grid) * len(sigma_grid) * len(eta_grid)
count = 0

for i_s0, s0 in enumerate(sigma0_grid):
    for i_sig, sig in enumerate(sigma_grid):
        for i_eta, eta in enumerate(eta_grid):
            dp = run_mc_dprime(s0, sig, eta)
            for isi in ISI_VALUES:
                results[isi][i_s0, i_sig, i_eta] = dp.get(isi, np.nan)
            count += 1
            if count % 50 == 0:
                print(f'  {count}/{total} done...')

print(f'Grid search complete: {count} configurations evaluated.')

## Heatmaps: d' as f(σ, η) for each ISI, sliced by σ₀

Each row = one ISI, each column = one σ₀ level.  
Color = d' value. This gives a compact overview of the full 3D parameter space.

In [ ]:
# Select a subset of sigma0 levels for columns
s0_slice_indices = [0, 2, 4, 6]  # 0.0, 0.25, 0.75, 1.5
s0_slice_vals = sigma0_grid[s0_slice_indices]

fig, axes = plt.subplots(len(ISI_VALUES), len(s0_slice_indices),
                         figsize=(4 * len(s0_slice_indices), 3.5 * len(ISI_VALUES)),
                         sharex=True, sharey=True)

vmin = 0
vmax = np.nanmax([results[isi] for isi in ISI_VALUES])
vmax = min(vmax, 5.0)  # cap for visual clarity

for row, isi in enumerate(ISI_VALUES):
    for col, i_s0 in enumerate(s0_slice_indices):
        ax = axes[row, col]
        data = results[isi][i_s0, :, :]  # [n_sigma, n_eta]
        im = ax.imshow(data, aspect='auto', origin='lower',
                       vmin=vmin, vmax=vmax, cmap='viridis',
                       extent=[eta_grid[0], eta_grid[-1],
                               sigma_grid[0], sigma_grid[-1]])
        if row == 0:
            ax.set_title(f'σ₀={sigma0_grid[i_s0]:.2f}', fontsize=11)
        if col == 0:
            ax.set_ylabel(f'ISI={isi}\nσ (diffusive)', fontsize=10)
        if row == len(ISI_VALUES) - 1:
            ax.set_xlabel('η (drift step)', fontsize=10)

fig.suptitle("d' across parameter space (rows=ISI, cols=σ₀)", fontsize=13, y=1.01)
fig.colorbar(im, ax=axes, shrink=0.6, label="d'")
fig.tight_layout()
fig.savefig(f'{SAVE_DIR}/dprime_heatmaps_sigma_eta.png', dpi=150, bbox_inches='tight')
plt.show()

## Heatmaps: d' as f(σ₀, η) for each ISI, sliced by σ

Complementary view: fix diffusive noise, sweep encoding noise vs drift.

In [ ]:
sig_slice_indices = [0, 2, 4, 6]  # 0.0, 0.05, 0.15, 0.3
sig_slice_vals = sigma_grid[sig_slice_indices]

fig, axes = plt.subplots(len(ISI_VALUES), len(sig_slice_indices),
                         figsize=(4 * len(sig_slice_indices), 3.5 * len(ISI_VALUES)),
                         sharex=True, sharey=True)

for row, isi in enumerate(ISI_VALUES):
    for col, i_sig in enumerate(sig_slice_indices):
        ax = axes[row, col]
        data = results[isi][:, i_sig, :]  # [n_sigma0, n_eta]
        im = ax.imshow(data, aspect='auto', origin='lower',
                       vmin=vmin, vmax=vmax, cmap='viridis',
                       extent=[eta_grid[0], eta_grid[-1],
                               sigma0_grid[0], sigma0_grid[-1]])
        if row == 0:
            ax.set_title(f'σ={sigma_grid[i_sig]:.3f}', fontsize=11)
        if col == 0:
            ax.set_ylabel(f'ISI={isi}\nσ₀ (encoding)', fontsize=10)
        if row == len(ISI_VALUES) - 1:
            ax.set_xlabel('η (drift step)', fontsize=10)

fig.suptitle("d' across parameter space (rows=ISI, cols=σ)", fontsize=13, y=1.01)
fig.colorbar(im, ax=axes, shrink=0.6, label="d'")
fig.tight_layout()
fig.savefig(f'{SAVE_DIR}/dprime_heatmaps_sigma0_eta.png', dpi=150, bbox_inches='tight')
plt.show()

## ISI decay summary: d'(ISI=0) − d'(ISI=16) as f(σ₀, η) at fixed σ

How much does memory degrade over time? Large positive values = strong ISI decay.

In [ ]:
# Compute ISI decay: d'(ISI=0) - d'(ISI=16)
decay = results[0] - results[16]  # [n_sigma0, n_sigma, n_eta]

fig, axes = plt.subplots(1, len(sig_slice_indices),
                         figsize=(4 * len(sig_slice_indices), 4),
                         sharex=True, sharey=True)

dmax = np.nanmax(np.abs(decay))
dmax = min(dmax, 3.0)

for col, i_sig in enumerate(sig_slice_indices):
    ax = axes[col]
    data = decay[:, i_sig, :]  # [n_sigma0, n_eta]
    im = ax.imshow(data, aspect='auto', origin='lower',
                   vmin=-dmax, vmax=dmax, cmap='RdBu_r',
                   extent=[eta_grid[0], eta_grid[-1],
                           sigma0_grid[0], sigma0_grid[-1]])
    ax.set_title(f'σ={sigma_grid[i_sig]:.3f}', fontsize=11)
    ax.set_xlabel('η (drift step)', fontsize=10)
    if col == 0:
        ax.set_ylabel('σ₀ (encoding)', fontsize=10)

fig.suptitle("ISI decay: d'(ISI=0) − d'(ISI=16)", fontsize=13, y=1.02)
fig.colorbar(im, ax=axes, shrink=0.8, label="Δd'")
fig.tight_layout()
fig.savefig(f'{SAVE_DIR}/isi_decay_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

## Save grid results for reuse

In [ ]:
np.savez(f'{SAVE_DIR}/grid_search_results.npz',
         sigma0_grid=sigma0_grid,
         sigma_grid=sigma_grid,
         eta_grid=eta_grid,
         isi_values=np.array(ISI_VALUES),
         **{f'dprime_isi{isi}': results[isi] for isi in ISI_VALUES})
print(f'Saved to {SAVE_DIR}/grid_search_results.npz')